<a href="https://colab.research.google.com/github/gianni04/projet-google-colab/blob/notebooks/Var%2BCvar%2BMonte_carlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# ==========================================
# CELLULE 1 : Installation et Imports
# ==========================================
!pip install yfinance scipy -q

import numpy as np
import pandas as pd
import yfinance as yf
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

In [37]:
# ==========================================
# CELLULE 2 : Moteur VaR & CVaR (Ultra-Robuste)
# ==========================================
def get_risk_metrics(tickers, weights, start_date="2020-01-01", end_date="2026-01-01", alpha=0.99, sims=50000):
    # Téléchargement
    df_raw = yf.download(tickers, start=start_date, end=end_date, progress=False)

    # Stratégie de secours pour trouver les prix
    if isinstance(df_raw.columns, pd.MultiIndex):
        # Si MultiIndex, on cherche 'Adj Close', sinon 'Close'
        if 'Adj Close' in df_raw.columns.levels[0]:
            data = df_raw['Adj Close']
        else:
            data = df_raw['Close']
    else:
        # Index plat
        if 'Adj Close' in df_raw.columns:
            data = df_raw['Adj Close']
        else:
            data = df_raw['Close']

    # Si c'est un seul ticker, on force la conversion en DataFrame pour éviter les erreurs de type
    if isinstance(data, pd.Series):
        name = tickers[0] if isinstance(tickers, list) else tickers
        data = data.to_frame(name=name)

    data = data.dropna(how='all')

    # Calcul des rendements log
    returns = np.log(data / data.shift(1)).dropna()

    # Alignement des colonnes (si yfinance a changé l'ordre)
    if isinstance(tickers, list) and len(tickers) > 1:
        returns = returns[tickers]

    port_returns = returns.dot(weights)

    # VaR & CVaR Historique
    sorted_returns = np.sort(port_returns)
    idx = int((1 - alpha) * len(sorted_returns))
    var_hist = sorted_returns[idx]
    cvar_hist = sorted_returns[:idx].mean()

    # VaR & CVaR Paramétrique
    cov_matrix = returns.cov()
    mu = returns.mean()
    port_mu = np.dot(weights, mu)
    port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    z_score = stats.norm.ppf(1 - alpha)
    var_param = port_mu + z_score * port_vol
    cvar_param = port_mu - port_vol * (stats.norm.pdf(z_score) / (1 - alpha))

    # Monte Carlo
    L = np.linalg.cholesky(cov_matrix)
    Z = np.random.standard_normal((len(tickers), sims))
    daily_shocks = mu.values[:, None] + np.dot(L, Z)
    sim_returns = np.dot(weights.T, daily_shocks)

    sorted_sim = np.sort(sim_returns)
    idx_sim = int((1 - alpha) * len(sorted_sim))
    var_mc = sorted_sim[idx_sim]
    cvar_mc = sorted_sim[:idx_sim].mean()

    return {
        "Historique": (var_hist, cvar_hist),
        "Paramétrique": (var_param, cvar_param),
        "Monte Carlo": (var_mc, cvar_mc)
    }

def print_results(name, metrics):
    if metrics is None: return
    print(f"--- RESULTATS : {name} ---")
    for method, (var, cvar) in metrics.items():
        print(f"{method:<15} -> VaR 99%: {var*100:>7.2f}% | CVaR 99%: {cvar*100:>7.2f}%")
    print("\n")

In [21]:
# ==========================================
# CELLULE 3 : Action Simple (UBS)
# ==========================================
res_ubs = get_risk_metrics(['UBS'], np.array([1.0]))
print_results("UBS Group (Seul)", res_ubs)

--- RESULTATS : UBS Group (Seul) ---
Historique      -> VaR 99%:   -5.64% | CVaR 99%:   -8.51%
Paramétrique    -> VaR 99%:   -4.86% | CVaR 99%:   -5.58%
Monte Carlo     -> VaR 99%:   -4.84% | CVaR 99%:   -5.62%




In [22]:
# ==========================================
# CELLULE 4 : Bitcoin Seul
# ==========================================
res_btc = get_risk_metrics(['BTC-USD'], np.array([1.0]))
print_results("Bitcoin (BTC)", res_btc)

--- RESULTATS : Bitcoin (BTC) ---
Historique      -> VaR 99%:   -8.91% | CVaR 99%:  -13.24%
Paramétrique    -> VaR 99%:   -7.42% | CVaR 99%:   -8.51%
Monte Carlo     -> VaR 99%:   -7.38% | CVaR 99%:   -8.44%




In [29]:
# ==========================================
# CELLULE 5 : Portefeuille 40% FR / 30% US / 20% Bonds / 10% ETH
# ==========================================
tickers_pf = ['MC.PA', 'SAN.PA', 'OR.PA', 'AIR.PA', 'AAPL', 'MSFT', 'NVDA', 'TLT', 'ETH-USD']
weights_pf = np.array([0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.2, 0.1])

res_pf = get_risk_metrics(tickers_pf, weights_pf)
print_results("Portefeuille Diversifié", res_pf)

--- RESULTATS : Portefeuille Diversifié ---
Historique      -> VaR 99%:   -3.24% | CVaR 99%:   -4.58%
Paramétrique    -> VaR 99%:   -2.75% | CVaR 99%:   -3.16%
Monte Carlo     -> VaR 99%:   -2.76% | CVaR 99%:   -3.20%




In [39]:
# ==========================================
# CELLULE 6 : Graphiques Interactifs (Hist, Param, Monte Carlo)
# ==========================================
import plotly.graph_objects as go
import numpy as np
import scipy.stats as stats
import pandas as pd
import yfinance as yf

def plot_interactive_distribution(tickers, weights, title="Distribution des Rendements", sims=50000):
    """Génère un graphique interactif Plotly avec VaR/CVaR pour les 3 méthodes."""

    # --- 1. Data Prep ---
    df_raw = yf.download(tickers, start="2020-01-01", end="2026-01-01", progress=False)
    if isinstance(df_raw.columns, pd.MultiIndex):
        if 'Adj Close' in df_raw.columns.get_level_values(0): data = df_raw['Adj Close']
        elif 'Adj Close' in df_raw.columns.get_level_values(1): data = df_raw.xs('Adj Close', level=1, axis=1)
        elif 'Close' in df_raw.columns.get_level_values(0): data = df_raw['Close']
        elif 'Close' in df_raw.columns.get_level_values(1): data = df_raw.xs('Close', level=1, axis=1)
    else:
        data = df_raw['Adj Close'] if 'Adj Close' in df_raw.columns else df_raw['Close']

    if isinstance(data, pd.Series):
        data = data.to_frame(name=tickers[0] if isinstance(tickers, list) else tickers)

    returns = np.log(data / data.shift(1)).dropna()
    if len(tickers) > 1: returns = returns[[t for t in tickers if t in returns.columns]]
    port_returns = returns.dot(weights)

    alpha = 0.99

    # --- 2. Calculs Historique ---
    sorted_ret = np.sort(port_returns)
    idx = int((1 - alpha) * len(sorted_ret))
    var_hist = sorted_ret[idx]
    cvar_hist = sorted_ret[:idx].mean()

    # --- 3. Calculs Paramétrique ---
    mu = returns.mean()
    cov = returns.cov()
    port_mu = np.dot(weights, mu)
    port_vol = np.sqrt(np.dot(weights.T, np.dot(cov, weights)))
    z = stats.norm.ppf(1 - alpha)
    var_param = port_mu + z * port_vol

    # --- 4. Calculs MONTE CARLO ---
    L = np.linalg.cholesky(cov)
    Z = np.random.standard_normal((len(tickers), sims))
    daily_shocks = mu.values[:, None] + np.dot(L, Z)
    sim_returns = np.dot(weights.T, daily_shocks)

    sorted_sim = np.sort(sim_returns)
    idx_sim = int((1 - alpha) * len(sorted_sim))
    var_mc = sorted_sim[idx_sim]
    cvar_mc = sorted_sim[:idx_sim].mean()

    # --- 5. Construction du Graphique Plotly ---
    fig = go.Figure()
    x_axis = np.linspace(port_returns.min() * 1.5, port_returns.max(), 500)

    # Trace 1: Histogramme réel
    fig.add_trace(go.Histogram(
        x=port_returns, histnorm='probability density',
        name="Historique (Réel)", marker_color='#1f77b4', opacity=0.6, nbinsx=120
    ))

    # Trace 2: Paramétrique (Normale)
    y_param = stats.norm.pdf(x_axis, port_mu, port_vol)
    fig.add_trace(go.Scatter(
        x=x_axis, y=y_param, mode='lines',
        name="Paramétrique (Normale)", line=dict(color='#ff7f0e', width=2)
    ))

    # Trace 3: Monte Carlo (KDE Densité)
    kde_mc = stats.gaussian_kde(sim_returns)
    y_mc = kde_mc.evaluate(x_axis)
    fig.add_trace(go.Scatter(
        x=x_axis, y=y_mc, mode='lines',
        name=f"Monte Carlo ({sims//1000}k sims)", line=dict(color='#2ca02c', width=2.5, dash='dash')
    ))

    # Lignes verticales VaR (On affiche que la VaR pour ne pas surcharger)
    fig.add_vline(x=var_hist, line_dash="solid", line_color="#1f77b4", annotation_text=f"VaR Hist ({var_hist*100:.2f}%)", annotation_position="top left")
    fig.add_vline(x=var_param, line_dash="dot", line_color="#ff7f0e", annotation_text=f"VaR Param ({var_param*100:.2f}%)", annotation_position="bottom right")
    fig.add_vline(x=var_mc, line_dash="dash", line_color="#2ca02c", annotation_text=f"VaR MC ({var_mc*100:.2f}%)", annotation_position="top right")

    fig.update_layout(
        title=f"<b>{title}</b><br><sup>Comparaison des 3 Modèles de Risque (Left Tail Focus)</sup>",
        xaxis_title="Rendements Journaliers",
        yaxis_title="Densité de Probabilité",
        template="plotly_dark",
        hovermode="x unified",
        legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99, bgcolor="rgba(0,0,0,0.5)")
    )

    fig.show()

# Exécution :
plot_interactive_distribution(['UBS'], np.array([1.0]), "Risque : Action UBS")
plot_interactive_distribution(['BTC-USD'], np.array([1.0]), "Risque : Bitcoin (BTC)")
plot_interactive_distribution(tickers_pf, weights_pf, "Risque : Portefeuille Diversifié")

In [40]:
# ==========================================
# CELLULE 7 : Monte Carlo "Expert" (Fan Chart) - CORRIGÉE
# ==========================================
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import yfinance as yf

def monte_carlo_fan_chart(tickers, weights, days=252, simulations=1000):
    # --- 1. Le bouclier de données (identique à la cellule 2) ---
    df_raw = yf.download(tickers, start="2020-01-01", end="2026-01-01", progress=False)

    if isinstance(df_raw.columns, pd.MultiIndex):
        if 'Adj Close' in df_raw.columns.levels[0]:
            data = df_raw['Adj Close']
        else:
            data = df_raw['Close']
    else:
        if 'Adj Close' in df_raw.columns:
            data = df_raw['Adj Close']
        else:
            data = df_raw['Close']

    if isinstance(data, pd.Series):
        name = tickers[0] if isinstance(tickers, list) else tickers
        data = data.to_frame(name=name)

    data = data.dropna(how='all')
    returns = np.log(data / data.shift(1)).dropna()

    if isinstance(tickers, list) and len(tickers) > 1:
        returns = returns[tickers]

    # --- 2. Mathématiques du GBM ---
    mu = returns.mean()
    cov = returns.cov()

    dt = 1/252
    port_mu = np.dot(weights, mu)
    port_vol = np.sqrt(np.dot(weights.T, np.dot(cov, weights)))

    # S(t) = S(0) * exp((mu - 0.5*sigma^2)*t + sigma*W(t))
    last_price = 100.0 # Base 100
    drift = (port_mu - 0.5 * port_vol**2) * dt
    diffusion = port_vol * np.sqrt(dt)

    paths = np.zeros((days, simulations))
    paths[0] = last_price

    for t in range(1, days):
        z = np.random.standard_normal(simulations)
        paths[t] = paths[t-1] * np.exp(drift + diffusion * z)

    # --- 3. Visualisation Interactive ---
    fig = go.Figure()

    # On trace 50 trajectoires aléatoires parmi les 1000 pour ne pas saturer le navigateur
    for i in range(50):
        fig.add_trace(go.Scatter(
            y=paths[:, i],
            mode='lines',
            line=dict(width=0.5, color='rgba(0,100,250,0.3)'),
            showlegend=False
        ))

    # On ajoute la médiane en blanc épais
    fig.add_trace(go.Scatter(
        y=np.percentile(paths, 50, axis=1),
        mode='lines',
        line=dict(color='white', width=3),
        name="Médiane"
    ))

    fig.update_layout(
        title="Monte Carlo : Simulation de trajectoires de prix (GBM)",
        xaxis_title="Jours de trading (Futur)",
        yaxis_title="Valeur du Portefeuille (Base 100)",
        template="plotly_dark",
        hovermode="x"
    )
    fig.show()

# Exécution
monte_carlo_fan_chart(tickers_pf, weights_pf)